# Initial Cleaning of the Merged Master Dataset

This notebook performs the first stage of cleaning and inspection for the merged thesis dataset.

It is designed to:
- load the merged dataset
- inspect column names, data types, and general data quality
- review columns in a structured way
- apply only basic and safe cleaning steps
- avoid model-specific preprocessing at this stage

The goal is to make the dataset easier to understand before any later feature engineering, imputation strategy, encoding, or modeling decisions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Load Dataset

The merged dataset combines price observations with Traficom-based features. A copy of the raw data is kept so that all cleaning steps remain explicit and easy to review.

In [ ]:
data_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df_raw = pd.read_csv(data_path, skipinitialspace=True)
df = df_raw.copy()

print(f"Dataset path: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

## Initial Overview

This first overview checks dataset size, column names, and the starting data types before any cleaning changes are applied.

In [ ]:
print("Original column names:")
print(df.columns.tolist())
print()

df.info()

## Duplicate Observation Warning

Some observations that may look like duplicates can represent the **same listing or part observed repeatedly over time**. In this project, these repeated observations should be preserved unless there is a very clear accidental exact-duplicate issue.

For that reason, this notebook only **reports** duplicate patterns. It does **not** automatically drop duplicate rows.

## Basic Cleaning

The cleaning below is intentionally conservative. It standardizes names and obvious formatting issues without making modeling decisions or deleting information.

In [ ]:
original_columns = df.columns.tolist()

# Standardize column names for easier handling.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

object_columns = df.select_dtypes(include=["object"]).columns.tolist()
for column in object_columns:
    df[column] = df[column].astype("string").str.strip()

# Replace empty strings with missing values after whitespace stripping.
df[object_columns] = df[object_columns].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

numeric_name_fragments = [
    "price",
    "mileage",
    "year",
    "age",
    "registered",
    "engine_cc",
    "power_kw",
    "mass_kg",
    "share",
    "span",
    "mid",
]

candidate_numeric_columns = [
    column
    for column in df.columns
    if any(fragment in column for fragment in numeric_name_fragments)
    and column not in {"product_id"}
]

for column in candidate_numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

date_like_columns = [column for column in df.columns if "date" in column]
for column in date_like_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

if "product_id" in df.columns:
    product_id_numeric = pd.to_numeric(df["product_id"], errors="coerce")
    df["product_id"] = product_id_numeric.astype("Int64").astype("string")

print("Column names before cleaning:")
print(original_columns)
print()
print("Column names after cleaning:")
print(df.columns.tolist())
print()
print(f"Object/string columns cleaned: {len(object_columns)}")
print(f"Numeric columns coerced where obvious: {len(candidate_numeric_columns)}")
print(f"Date-like columns parsed where obvious: {len(date_like_columns)}")

## Column Summary

A compact summary table helps identify the role and quality of each column. The conceptual column type is partly heuristic, so it should be treated as a review aid rather than a final classification.

In [ ]:
def conceptual_column_type(series: pd.Series) -> str:
    column_name = series.name.lower()

    if column_name == "price":
        return "target"
    if column_name.endswith("_id") or column_name == "product_id":
        return "id"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if any(token in column_name for token in ["date", "time"]):
        return "date"
    if any(token in column_name for token in ["name", "description", "comment", "text", "number"]):
        return "text"
    if series.nunique(dropna=True) <= 30:
        return "categorical"
    return "text"


def sample_values(series: pd.Series, max_values: int = 5) -> str:
    values = series.dropna().astype(str).unique().tolist()[:max_values]
    return ", ".join(values)


column_summary = pd.DataFrame(
    {
        "column_name": df.columns,
        "dtype": [str(df[column].dtype) for column in df.columns],
        "conceptual_type": [conceptual_column_type(df[column]) for column in df.columns],
        "missing_count": [int(df[column].isna().sum()) for column in df.columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        "unique_values": [int(df[column].nunique(dropna=True)) for column in df.columns],
        "sample_values": [sample_values(df[column]) for column in df.columns],
    }
).sort_values(["conceptual_type", "column_name"]).reset_index(drop=True)


display(column_summary)

## Column-by-Column Inspection

The next cells inspect the dataset by column type. This is useful for spotting unusual ranges, unexpected values, and columns that may need more careful treatment later.

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_summary = pd.DataFrame(
    {
        "column_name": numeric_columns,
        "missing_count": [int(df[column].isna().sum()) for column in numeric_columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in numeric_columns],
        "min": [df[column].min() for column in numeric_columns],
        "max": [df[column].max() for column in numeric_columns],
        "mean": [df[column].mean() for column in numeric_columns],
        "median": [df[column].median() for column in numeric_columns],
    }
).sort_values("column_name")

print(f"Numeric columns: {len(numeric_columns)}")
display(numeric_summary)

In [ ]:
categorical_columns = [
    column
    for column in df.columns
    if df[column].dtype in ["object", "string", "category", "boolean"]
]

print(f"Categorical/text-like columns selected for quick review: {len(categorical_columns)}")

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(f"Missing values: {df[column].isna().sum():,} ({df[column].isna().mean() * 100:.2f}%)")
    display(df[column].value_counts(dropna=False).head(10).rename("count").to_frame())


In [ ]:
date_columns = [column for column in df.columns if pd.api.types.is_datetime64_any_dtype(df[column])]

if date_columns:
    date_summary = pd.DataFrame(
        {
            "column_name": date_columns,
            "missing_count": [int(df[column].isna().sum()) for column in date_columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in date_columns],
            "min_date": [df[column].min() for column in date_columns],
            "max_date": [df[column].max() for column in date_columns],
        }
    ).sort_values("column_name")
    display(date_summary)
else:
    print("No date columns were parsed successfully in the basic cleaning step.")

## Duplicate Checks

These checks are purely diagnostic. They help assess whether there are exact row duplicates or repeated identifiers, but no rows are removed automatically.

In [ ]:
full_row_duplicates = int(df.duplicated().sum())
print(f"Fully duplicated rows: {full_row_duplicates:,}")

id_columns = [column for column in df.columns if column.endswith("_id") or column == "product_id"]

if id_columns:
    duplicate_id_summary = pd.DataFrame(
        {
            "column_name": id_columns,
            "duplicate_values": [int(df[column].duplicated(keep=False).sum()) for column in id_columns],
            "unique_values": [int(df[column].nunique(dropna=True)) for column in id_columns],
            "missing_count": [int(df[column].isna().sum()) for column in id_columns],
        }
    ).sort_values("column_name")
    display(duplicate_id_summary)
else:
    print("No ID-like columns were detected.")

## Missing Values Review

Missing values are reviewed after the basic cleaning step so that empty strings and parsing failures are already reflected in the totals.

In [ ]:
missing_values_review = (
    pd.DataFrame(
        {
            "column_name": df.columns,
            "missing_count": [int(df[column].isna().sum()) for column in df.columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        }
    )
    .sort_values(["missing_percentage", "missing_count"], ascending=[False, False])
    .reset_index(drop=True)
)

print("Columns with the highest missingness:")
display(missing_values_review.head(20))

## Save Cleaned Dataset

The file saved here is still an **initial cleaned version**. It reflects only conservative formatting and type fixes carried out in this notebook.

In [ ]:
output_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/price_traficom_merged_cleaned_initial.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Initial cleaned dataset saved to: {output_path}")